In [1]:
# ============================================================
# NOTEBOOK 08 — BLOCK A — attach exposure
# ============================================================
import pandas as pd, numpy as np

rated = pd.read_parquet('data/processed/test_rated.parquet')
ead   = pd.read_parquet('data/processed/test_raw.parquet', columns=['id','loan_amnt'])
port  = rated.merge(ead, on='id', how='left')

print(f"{len(port):,} loans | merge nulls {port['loan_amnt'].isna().sum()}")
print(f"Total exposure: ${port['loan_amnt'].sum()/1e9:.2f}bn")
print(port.groupby('rating', observed=False)['loan_amnt'].agg(
    n='size', total='sum', avg='mean').assign(
    pct=lambda x: (x['total']/x['total'].sum()*100).round(1)).round(0).to_string())

375,546 loans | merge nulls 0
Total exposure: $5.50bn
            n         total      avg   pct
rating                                    
A       14959  2.255108e+08  15075.0   4.0
B       39216  5.844449e+08  14903.0  11.0
C       75861  1.063692e+09  14022.0  19.0
D       91656  1.194907e+09  13037.0  22.0
E       78187  1.093722e+09  13989.0  20.0
F       44143  7.689803e+08  17420.0  14.0
G       31524  5.673449e+08  17997.0  10.0


In [2]:
# ============================================================
# BLOCK B — three scenarios on the 2015 book
# Baseline : scorecard's predicted PD
# S1       : parallel +3.01pp (naive stress test)
# S2       : observed 2015 grade-specific gaps (same count-weighted
#            average shock as S1 — only the SHAPE differs)
# LGD is assumed, not estimated: no recovery data in this dataset.
# ============================================================
GAPS = {'A': -0.0059, 'B': 0.0030, 'C': 0.0117, 'D': 0.0195,
        'E': 0.0329, 'F': 0.0646, 'G': 0.0982}
SHOCK = 0.0301

g = port.groupby('rating', observed=False).agg(
    ead=('loan_amnt', 'sum'), pd_base=('pd_sc', 'mean'), n=('id', 'size'))
g['pd_s1'] = g['pd_base'] + SHOCK
g['pd_s2'] = g['pd_base'] + pd.Series(GAPS)

E = g['ead'].sum()
print(f"Count-weighted PD  : {port['target'].mean():.4f}")
print(f"Dollar-weighted PD : {(g['ead'] * port.groupby('rating', observed=False)['target'].mean()).sum() / E:.4f}\n")

for lgd in (0.50, 0.65, 0.80):
    el = {s: (g['ead'] * g[f'pd_{s}']).sum() * lgd / 1e6
          for s in ('base', 's1', 's2')}
    print(f"LGD {lgd:.0%} | base ${el['base']:,.0f}m | "
          f"S1 ${el['s1']:,.0f}m | S2 ${el['s2']:,.0f}m | "
          f"S2−S1 ${el['s2']-el['s1']:,.0f}m ({(el['s2']/el['s1']-1)*100:+.1f}%)")

print("\nLoss by grade, S2, LGD 65%:")
out = (g[['n','ead','pd_base','pd_s2']]
       .assign(el_m=lambda x: x['ead'] * x['pd_s2'] * 0.65 / 1e6,
               pct_of_loss=lambda x: (x['ead']*x['pd_s2'] / (g['ead']*g['pd_s2']).sum()*100).round(1),
               pct_of_ead=lambda x: (x['ead']/E*100).round(1)))
print(out[['pct_of_ead','pd_base','pd_s2','el_m','pct_of_loss']].round(4).to_string())

Count-weighted PD  : 0.2019
Dollar-weighted PD : 0.2117

LGD 50% | base $493m | S1 $576m | S2 $582m | S2−S1 $6m (+1.0%)
LGD 65% | base $641m | S1 $749m | S2 $757m | S2−S1 $8m (+1.0%)
LGD 80% | base $789m | S1 $922m | S2 $931m | S2−S1 $10m (+1.0%)

Loss by grade, S2, LGD 65%:
        pct_of_ead  pd_base   pd_s2      el_m  pct_of_loss
rating                                                    
A         4.100000   0.0394  0.0335    4.9064          0.6
B        10.600000   0.0664  0.0694   26.3607          3.5
C        19.299999   0.1006  0.1123   77.6136         10.3
D        21.700001   0.1440  0.1635  127.0083         16.8
E        19.900000   0.2005  0.2334  165.9594         21.9
F        14.000000   0.2795  0.3441  171.9743         22.7
G        10.300000   0.3978  0.4960  182.9094         24.2


In [3]:
# ============================================================
# BLOCK C — scenario table, saved
# S2 is not hypothetical: it is realised 2015 minus predicted.
# LGD is assumed (no recovery data in this dataset).
# ============================================================
scen = pd.DataFrame({
    lgd: {'base_EL_m':  (g['ead']*g['pd_base']).sum()*lgd/1e6,
          'S1_EL_m':    (g['ead']*g['pd_s1']).sum()*lgd/1e6,
          'S2_EL_m':    (g['ead']*g['pd_s2']).sum()*lgd/1e6,
          'unexpected_m': (g['ead']*(g['pd_s2']-g['pd_base'])).sum()*lgd/1e6}
    for lgd in (0.50, 0.65, 0.80)}).round(0)
scen.columns = [f'LGD_{int(c*100)}' for c in scen.columns]
print(scen.to_string())
scen.to_csv('data/processed/scenario_table.csv')

conc = out[['pct_of_ead','pct_of_loss']].copy()
conc.to_csv('data/processed/loss_concentration.csv')
print(f"\nF+G: {conc.loc[['F','G'],'pct_of_ead'].sum():.1f}% of EAD, "
      f"{conc.loc[['F','G'],'pct_of_loss'].sum():.1f}% of loss")

              LGD_50  LGD_65  LGD_80
base_EL_m      493.0   641.0   789.0
S1_EL_m        576.0   749.0   922.0
S2_EL_m        582.0   757.0   931.0
unexpected_m    89.0   115.0   142.0

F+G: 24.3% of EAD, 46.9% of loss
